# 02 - The `llima` CLI, Subcommand By Subcommand

**Why.** LLiMa is how you get a model onto the board and how you smoke-test it without writing code.
This notebook explains every subcommand, shows its **real captured `--help` output**, and says
**what happens on disk**. The cheap probes (`--help`, `list`, `search`) were captured on the board on
2026-07-09 and are embedded as text below - so this whole notebook is safe to read without touching
hardware.

The `pull` / `run` / `benchmark-server` commands are heavy: they download gigabytes or run a model.
They are **DOCUMENTED, NOT EXECUTED** here. Cells that would run them are Python strings, not shell
calls, so they cannot fire by accident.

## Where the CLI lives and how to reach it

`llima` is at **`/usr/bin/llima`** on the DevKit (board `192.168.135.203`, user `sima`). It is not in
the SDK container.

**Human, real terminal:**
```bash
dk /bin/bash    # then run llima commands interactively on the board
```

**Automation / CI (no TTY - `dk` hangs, use ssh with a timeout):**
```bash
timeout 60 ssh -o BatchMode=yes sima@192.168.135.203 'llima --help'
```

> Note captured 2026-07-09: `llima` requires PyYAML on the board's Python. It was already installed
> by the owner, so the CLI works. If you ever see `ModuleNotFoundError: No module named 'yaml'`,
> install `python3-yaml` (a persistent, owner-authorized change) before using the CLI.

## The top-level surface

Captured `llima --help`:

```text
usage: llima [-h] {run,search,pull,list,rm,benchmark-server} ...

Llima unified CLI

positional arguments:
  {run,search,pull,list,rm,benchmark-server}
    run                 Run a model
    search              Search remote models
    pull                Download a model
    list                List local models
    rm                  Remove a local model
    benchmark-server    Start a server for benchmarking a model
```

**Correctness trap #1:** the serve subcommand is **`benchmark-server`, NOT `serve`**. There is no
`llima serve`. Typing `llima serve` will error.

In [ ]:
# Safe reference cell: a map of the six subcommands. Prints only.
SUBCOMMANDS = {
    "search":           ("cheap/safe", "Search the remote catalog"),
    "list":             ("cheap/safe", "List models already on the board"),
    "pull":             ("HEAVY",      "Download a model directory (gigabytes)"),
    "run":              ("HEAVY",      "Load and run a model (LLM/VLM/ASR)"),
    "rm":               ("cheap",      "Delete a local model directory"),
    "benchmark-server": ("HEAVY",      "Start an HTTP benchmarking server (NOT `serve`)"),
}
for name, (cost, desc) in SUBCOMMANDS.items():
    print(f"llima {name:16s} [{cost:10s}] {desc}")

## `llima search` - browse the remote catalog (cheap, safe)

Captured `llima search --help`:

```text
usage: llima search [-h] [term]

positional arguments:
  term        Search term (optional)
```

With no term it lists the full remote catalog; with a term it filters. **On disk: nothing.** It only
queries the remote catalog and prints IDs. Run it before `pull` to get the exact model ID string.

```bash
llima search              # full catalog (36 models, captured 2026-07-09)
llima search Qwen3        # filter to Qwen3 family
```

Filtering locally against the captured catalog (safe cell):

In [ ]:
CATALOG = """llava-1.5-7b-hf-a16w4 Llama-3.1-8B-Instruct-a16w4 Llama-3.2-3B-Instruct-a16w4
Phi-3.5-mini-instruct-a16w4 paligemma-3b-pt-224-a16w8 gemma-3-4b-it-a16w4 Llama-2-7b-chat-hf-a16w4
whisper-small-a16w8 gemma-3-1b-it-a16w4 Mistral-7B-Instruct-v0.3-a16w4 gemma3-siglip448-a16w4
Qwen2.5-0.5B-Instruct-GPTQ-a16w4 Qwen2.5-7B-Instruct-GPTQ-a16w4 Qwen3-0.6B-GPTQ-a16w4
Qwen3-1.7B-GPTQ-a16w4 Qwen3-4B-Instruct-2507-GPTQ-a16w4 Qwen3-8B-GPTQ-a16w4 LFM2-VL-450M-a16w4
LFM2-VL-1.6B-a16w4 LFM2-VL-3B-a16w4 Qwen2.5-VL-3B-Instruct-GPTQ-a16w4 Qwen3-VL-4B-Instruct-GPTQ-a16w4
Qwen3-VL-8B-Instruct-GPTQ-a16w4 LFM2-350M-a16w4 LFM2-1.2B-a16w4 LFM2-2.6B-a16w4
Qwen2.5-VL-7B-Instruct-GPTQ-a16w4 Qwen3-VL-2B-Instruct-GPTQ-a16w4 gemma-4-E2B-it-GPTQ-a16w4
gemma-4-E4B-it-GPTQ-a16w4 LFM2.5-230M-a16w4 LFM2.5-350M-a16w4 LFM2.5-1.2B-Instruct-a16w4
LFM2.5-1.2B-Thinking-a16w4 LFM2.5-VL-450M-a16w4 LFM2.5-VL-1.6B-a16w4""".split()

term = "Qwen3"
print(f"llima search {term}  ->")
for m in CATALOG:
    if term.lower() in m.lower():
        print(" ", m)

## `llima list` - what is already on the board (cheap, safe)

**On disk: nothing changes.** It reads the local model directory and prints the IDs present. Captured
`llima list` (2026-07-09):

```text
Qwen3-4B-Instruct-2507-GPTQ-a16w4
Qwen3-VL-4B-Instruct-GPTQ-a16w4
whisper-small-a16w8
```

These three (LLM / VLM / ASR) are the models every notebook in `../02-run-llm-vlm/` targets. Run
`llima list` first to confirm they are present before running a model.

## `df -h /` before you pull (a required precheck)

Captured `df -h /` on the board (2026-07-09):

```text
Filesystem      Size  Used Avail Use% Mounted on
/dev/root        14G  7.3G  5.9G  56% /
```

Only ~5.9 GB free. **Always run this before `llima pull`.** A GenAI model directory can be several
GB; a pull that runs the disk to 0 leaves the board in a bad state. If free space is tight, pick the
smallest viable model or `llima rm` something first.

## `llima pull` - download a model (HEAVY - documented, not executed)

Captured `llima pull --help`:

```text
usage: llima pull [-h] model

positional arguments:
  model       Model ID
```

**What happens on disk:** `pull` downloads the model (weights already quantized/compiled for Modalix)
and writes a **model directory** under `/media/nvme/llima/models/<model-id>`. That directory is what
`pyneat.genai` and `llima run` later load. The download is gigabytes and can take minutes.

**Not needed for this folder** - the three happy-path models are already present. If you pull fresh,
`df -h /` first, then prefer a small variant.

In [ ]:
# DOCUMENTED, NOT EXECUTED - these are strings, running this cell only prints them.
PULL_PRECHECK = "df -h /"                      # ALWAYS run this first (cheap)
PULL_SMALLEST_LLM = "llima pull LFM2.5-230M-a16w4"   # smallest fresh LLM
PULL_SMALLEST_VLM = "llima pull LFM2-VL-450M-a16w4"  # smallest fresh VLM
# Result on disk: /media/nvme/llima/models/<model-id>/
print("precheck        :", PULL_PRECHECK)
print("pull small LLM  :", PULL_SMALLEST_LLM)
print("pull small VLM  :", PULL_SMALLEST_VLM)
print("lands at        : /media/nvme/llima/models/<model-id>/")

## `llima run` - quick interactive test (HEAVY - documented, not executed)

Captured `llima run --help`:

```text
usage: llima run [-h] [--mode {cli,web}] [--stt_model_path STT_MODEL_PATH]
                 [--system_prompt SYSTEM_PROMPT | --system_prompt_file ... |
                  --chat_template ... | --chat_template_file ...]
                 [--parallel_load | --no-parallel_load] [--log_level LOG_LEVEL]
                 model

positional arguments:
  model                 Model path or model ID
options:
  --mode {cli,web}
  --stt_model_path STT_MODEL_PATH   Path to the elf files for speech-to-text model.
  --system_prompt / --system_prompt_file / --chat_template / --chat_template_file
  --parallel_load, --no-parallel_load
  --log_level LOG_LEVEL
```

`llima run` loads the model and gives you an interactive prompt (`--mode cli`) or a web UI
(`--mode web`) - a no-code smoke test. **On disk:** nothing new is written; it loads an existing model
directory into memory.

**Correctness trap #2 - ASR.** A speech-to-text model is run by passing its ELF directory explicitly
with **`--stt_model_path <elf-dir>`**. Text/vision models do not need this flag; ASR does.

In [ ]:
# DOCUMENTED, NOT EXECUTED - strings only.
# LLM smoke test (text):
RUN_LLM = "llima run --mode cli Qwen3-4B-Instruct-2507-GPTQ-a16w4"

# LLM with a system prompt:
RUN_LLM_SYS = ('llima run --mode cli '
               '--system_prompt \'You are concise.\' '
               'Qwen3-4B-Instruct-2507-GPTQ-a16w4')

# ASR smoke test (audio) - note the --stt_model_path <elf-dir> (trap #2):
RUN_ASR = ("llima run --stt_model_path "
           "/media/nvme/llima/models/whisper-small-a16w8 "
           "whisper-small-a16w8")

for label, cmd in [("LLM", RUN_LLM), ("LLM+system", RUN_LLM_SYS), ("ASR", RUN_ASR)]:
    print(f"{label:11s}: {cmd}")

## `llima rm` - delete a local model (cheap)

Removes a model directory under `/media/nvme/llima/models/`, freeing disk. Use it to make room before
a `pull` when `df -h /` is tight. **On disk:** deletes that model's directory. It does not touch the
remote catalog, so you can `pull` it again later.

```bash
llima rm <model-id>
```

Do not remove the three happy-path models unless you intend to re-pull them - the run notebooks
depend on them.

## `llima benchmark-server` - throughput server (HEAVY - documented, not executed)

**Correctness trap #1 again:** the subcommand is `benchmark-server`, **not** `serve`.

Captured `llima benchmark-server --help`:

```text
usage: llima benchmark-server [-h] [--port PORT] model

positional arguments:
  model        Model ID or path
options:
  --port PORT  Listening port
```

Starts an HTTP server that hosts the model for throughput measurement. **On disk:** nothing; it holds
the model in memory and serves requests. This is the LLiMa CLI's benchmarking front door.

> Distinct from the application server `pyneat.genai.GenAIServer` (OpenAI-compatible endpoints on
> port 9998 by default), which you build into your own app - that is covered in the `05-genai-server`
> track, not here.

In [ ]:
# DOCUMENTED, NOT EXECUTED - string only.
BENCH = "llima benchmark-server --port 8080 Qwen3-4B-Instruct-2507-GPTQ-a16w4"
print(BENCH)
print("# WRONG (there is no `serve` subcommand):")
print("# llima serve Qwen3-4B-Instruct-2507-GPTQ-a16w4   <-- errors")

## Expected output

The safe cells print:

- the six-row subcommand map,
- the `Qwen3` filter over the captured catalog:

```text
llima search Qwen3  ->
  Qwen3-0.6B-GPTQ-a16w4
  Qwen3-1.7B-GPTQ-a16w4
  Qwen3-4B-Instruct-2507-GPTQ-a16w4
  Qwen3-8B-GPTQ-a16w4
  Qwen2.5-... (any ID containing "qwen3" case-insensitively)
```

- the pull/run/benchmark command strings (printed, never executed).

**When you run the heavy commands manually on the board**, expect:

- `llima pull <id>` - a progress/download log, ending with the model directory present under
  `/media/nvme/llima/models/<id>` (confirm with `llima list`).
- `llima run --mode cli <id>` - a prompt where you type text and the model replies.
- `llima benchmark-server --port 8080 <id>` - a line indicating the server is listening on the port.

## Recap

- Six subcommands: `search`, `list`, `pull`, `run`, `rm`, `benchmark-server`.
- **Cheap/safe:** `search`, `list`, `rm`. **Heavy:** `pull`, `run`, `benchmark-server`.
- **Trap #1:** it is `benchmark-server`, not `serve`.
- **Trap #2:** ASR runs via `llima run --stt_model_path <elf-dir> <model>`.
- **`df -h /` before every `pull`** - the board has ~5.9 GB free.
- `pull` writes to `/media/nvme/llima/models/<id>/`; that directory is what the Python API loads next.

Next: `../02-run-llm-vlm/01_run_llm.ipynb` runs the LLM from Python.

## References

- Captured CLI output: on-board `llima --help` / subcommand `--help` / `list` / `search` (2026-07-09)
- `/workspace/core/tutorials/019_run_an_llm/README.md` (`llima pull` usage)
- `/workspace/core/docs/develop-apps/development-workflow/genai-model.mdx`